In [11]:
from ultralytics import YOLO


model = YOLO(model="yolo11s.pt")

In [23]:
import json
import shutil
from pathlib import Path
from typing import Any


# 1. 경로 설정
PROJECT_ROOT = Path("/home/jongha/projects/FGVC-DINO-v3")
ANNOTATION_JSON_PATH = PROJECT_ROOT / "data/Iron-Scraps/detection/instance_default.json"
SRC_IMAGE_DIR = PROJECT_ROOT / "data/Iron-Scraps/segmentation/task_232_dataset/images/default"
DEST_DIR = PROJECT_ROOT / "data" / "Iron-Scraps" / "detection"


# 2. YOLO 전용 폴더 구조 정의
SUB_DIRS = [
    DEST_DIR / "images/train",
    DEST_DIR / "images/test",
    DEST_DIR / "labels/train",
    DEST_DIR / "labels/test",
]


# 3. 폴더 생성
for sub_dir in SUB_DIRS:
    sub_dir.mkdir(parents=True, exist_ok=True)

print("YOLO 데이터셋 폴더 구조가 성공적으로 준비되었습니다.")

YOLO 데이터셋 폴더 구조가 성공적으로 준비되었습니다.


In [24]:
# 1. COCO 어노테이션 파일 읽기
with open(ANNOTATION_JSON_PATH, "r", encoding="utf-8") as f:
    annotation_data = json.load(f)

In [25]:
annotation_data["annotations"]

[{'id': 1,
  'image_id': 1,
  'category_id': 2,
  'segmentation': [[1502.46,
    765.53,
    1479.65,
    740.96,
    1462.98,
    721.67,
    1455.96,
    693.6,
    1456.84,
    665.53,
    1476.14,
    657.63,
    1517.37,
    653.25,
    1536.67,
    665.53,
    1550.7,
    683.95,
    1555.09,
    709.39,
    1550.7,
    731.32,
    1532.28,
    749.74,
    1512.98,
    752.37]],
  'area': 7981.0,
  'bbox': [1455.96, 653.25, 99.13, 112.28],
  'iscrowd': 0,
  'attributes': {'occluded': False}},
 {'id': 2,
  'image_id': 1,
  'category_id': 2,
  'segmentation': [[3520.88,
    1498.86,
    3538.42,
    1469.04,
    3553.33,
    1452.37,
    3597.19,
    1440.96,
    3619.12,
    1444.47,
    3675.26,
    1479.56,
    3784.91,
    1547.11,
    3753.33,
    1568.16,
    3720.0,
    1594.47,
    3665.61,
    1610.26,
    3615.61,
    1587.46,
    3581.4,
    1567.28,
    3576.14,
    1544.47,
    3541.93,
    1510.26]],
  'area': 24144.0,
  'bbox': [3520.88, 1440.96, 264.03, 169.3],
  'i

In [26]:
# 2. 이미지 파일명 기준으로 정렬하여 연속 프레임 그룹 유지
images: list[dict[str, Any]] = sorted(annotation_data["images"], key=lambda x: x["file_name"])
total_count = len(images)


# 3. 8:2 비율로 순차 분할점 계산
split_index = int(total_count * 0.8)
train_images = images[:split_index]
val_images = images[split_index:]


print(f"전체 이미지 수: {total_count}장")
print(f"└ Train 세트: {len(train_images)}장 (앞 80%)")
print(f"└ Val 세트: {len(val_images)}장 (뒤 20%)")

전체 이미지 수: 6771장
└ Train 세트: 5416장 (앞 80%)
└ Val 세트: 1355장 (뒤 20%)


In [27]:
val_images

[{'id': 2163,
  'width': 3840,
  'height': 2160,
  'file_name': 'hyundaisteel-incheon-80t-inspection-danger/0-raw/base/5_1115462.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 2164,
  'width': 3840,
  'height': 2160,
  'file_name': 'hyundaisteel-incheon-80t-inspection-danger/0-raw/base/5_1117682.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 2165,
  'width': 3840,
  'height': 2160,
  'file_name': 'hyundaisteel-incheon-80t-inspection-danger/0-raw/base/5_1121224.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 2166,
  'width': 3840,
  'height': 2160,
  'file_name': 'hyundaisteel-incheon-80t-inspection-danger/0-raw/base/5_1126893.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 2167,
  'width': 3840,
  'height': 2160,
  'file_name': 'hyundaisteel-incheon-80t-inspection-danger/0-raw/base/5_1127922.jpg',
  'license': 0,
 

In [28]:
def convert_coco_to_yolo_bbox(
    coco_bbox: list[float], 
    img_w: int, 
    img_h: int
) -> tuple[float, float, float, float]:
    """COCO Bounding Box 좌표를 YOLO 포맷의 상대 좌표로 변환합니다."""
    x_min, y_min, w, h = coco_bbox
    
    # 1. 중심 좌표 계산
    x_center = x_min + (w / 2.0)
    y_center = y_min + (h / 2.0)
    
    # 2. 이미지 가로/세로 길이에 맞춰 0~1 사이 값으로 정규화
    rx = x_center / img_w
    ry = y_center / img_h
    rw = w / img_w
    rh = h / img_h
    
    return rx, ry, rw, rh


In [29]:
# 1. 이미지 매핑 및 어노테이션 그룹화 준비
annotations = annotation_data["annotations"]
image_to_annotations: dict[int, list[dict[str, Any]]] = {}
for annotation in annotations:
    img_id = annotation["image_id"]
    image_to_annotations.setdefault(img_id, []).append(annotation)

def process_split(split_images: list[dict[str, Any]], split_name: str) -> None:
    """각 분할 세트(train/val)의 이미지를 복사하고 YOLO 텍스트 라벨을 생성합니다."""
    for img_info in split_images:
        img_id = img_info["id"]
        file_name = img_info["file_name"]
        w, h = img_info["width"], img_info["height"]
        
        # 원본 이미지 경로와 대상 이미지 경로 설정
        src_image_path = SRC_IMAGE_DIR / file_name
        dest_image_path = DEST_DIR / "images" / split_name / src_image_path.name
        
        # 1. 이미지 물리 파일 복사 (존재할 경우에만)
        if src_image_path.exists():
            shutil.copy(src_image_path, dest_image_path)
        
        # 2. YOLO 라벨 파일 작성 (.txt)
        txt_name = Path(file_name).stem + ".txt"
        dest_label_path = DEST_DIR / "labels" / split_name / txt_name
        
        img_anns = image_to_annotations.get(img_id, [])
        yolo_lines = []
        for ann in img_anns:
            # COCO 카테고리 ID를 0부터 시작하는 클래스 인덱스로 매핑
            # 예: COCO의 cut (ID 2) -> YOLO 클래스 0, danger (ID 3) -> YOLO 클래스 1
            class_id = ann["category_id"] - 2 # (2, 3을 0, 1로 변환)
            
            # 좌표 변환
            rx, ry, rw, rh = convert_coco_to_yolo_bbox(ann["bbox"], w, h)
            yolo_lines.append(f"{class_id} {rx:.6f} {ry:.6f} {rw:.6f} {rh:.6f}")
        
        # 텍스트 파일 저장
        with open(dest_label_path, "w", encoding="utf-8") as lf:
            lf.write("\n".join(yolo_lines))

# 2. 실행
print("데이터셋 변환 및 복사 작업을 시작합니다 (시간이 다소 소요될 수 있습니다)...")
process_split(train_images, "train")
process_split(val_images, "test")
print("모든 이미지 복사 및 YOLO 라벨 파일 구축이 완료되었습니다!")


데이터셋 변환 및 복사 작업을 시작합니다 (시간이 다소 소요될 수 있습니다)...
모든 이미지 복사 및 YOLO 라벨 파일 구축이 완료되었습니다!


In [34]:
# 2. 모델 학습 시작
results = model.train(
    data="yolo_data.yaml",    # 설정 파일 경로
    epochs=50,                # 총 epoch 수
    imgsz=640,                # 이미지 입력 크기 (YOLO 기본값 640)
    batch=16,                 # 배치 사이즈 (메모리에 맞춰 조절)
    device=0,                 # GPU 0번 사용 (CPU 사용 시 'cpu')
    workers=4                 # 데이터 로더 멀티프로세스 개수
)


Ultralytics 8.4.53 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

KeyboardInterrupt: 